## Pseudo-XRR from 1D GIXOS linecuts (OPLS / NSLS-II 12ID)

Step-by-step workflow analogous to `pXRR_step2.ipynb`, using the new pxrr
(extended capillary wave model) API and the OPLS metadata configuration
(`opls_1d/gixos-process_config_1d.yaml`).

### 1. Imports

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.constants import pi

from pxrr.eCWM import *
from pxrr.data_io import (
    load_gixos_from_meta,
    save_metadata_yaml,
    export_gixos_nxs,
    load_gixos_nxs,
    export_orso,
)
from pxrr.gixos import (
    binning_GIXOS_tt,
    remove_negative_2theta,
    GIXOS_th2q,
    GIXOS_background_corr,
    GIXOS_qxy_dependence,
    GIXOS2R,
)
from pxrr.helpers import make_filename

%matplotlib inline

ModuleNotFoundError: No module named 'numpy'

### 2. Load configuration and data

Load sample and chamber-background GIXOS data directly from the OPLS YAML
configuration. All metadata is attached to the GIXOS dictionaries under
`["metadata"]`; the YAML file is no longer needed after this point.

In [ ]:
metadata_file = "./opls_1d/gixos-process_config_1d.yaml"
GIXOSdata, GIXOSbkg = load_gixos_from_meta(metadata_file)

### 3. Corrections

- vertical binning in `tt` (beta)
- remove negative 2theta rows
- convert angular axes to Q

In [ ]:
GIXOSdata = binning_GIXOS_tt(GIXOSdata)
GIXOSbkg  = binning_GIXOS_tt(GIXOSbkg)

GIXOSdata = remove_negative_2theta(GIXOSdata)
GIXOSbkg  = remove_negative_2theta(GIXOSbkg)

GIXOSdata_q = GIXOS_th2q(GIXOSdata)
GIXOSbkg_q  = GIXOS_th2q(GIXOSbkg)

### 4. Background subtraction

Subtract chamber background and a constant bulk background determined from
the high-Qz average (`bulkbkg_mode=0`, `bulkbkg_const_mode=1`).

In [ ]:
GIXOS_ana = GIXOS_background_corr(
    GIXOSdata_q,
    GIXOSbkg_q,
    bulkbkg_mode=0,
    bulkbkg_const_mode=1,
    bulkbkg_const_qz_lb=0.7,
    plot=True,
)

### 5. Save / reload corrected GIXOS (NeXus HDF5)

Store the corrected GIXOS dictionary (data + metadata) so the analysis can
be resumed later. The reloaded copy `GIXOS_back` is only shown for demo.

In [ ]:
outgixosfile = make_filename(GIXOS_ana["metadata"], suffix="gixos.h5")
export_gixos_nxs(GIXOS_ana, outgixosfile)

GIXOS_back = load_gixos_nxs(outgixosfile)
print("reloaded keys:", list(GIXOS_back.keys()))

### 6. Qxy dependence fit

Fit the bending modulus `kappa` from the Qxy dependence of the diffuse
intensity at the Qz values listed in `metadata['dependency']['qz_selected']`.
Subsequent operations write back into `GIXOS_ana` (shared dict).

In [ ]:
_, qxy_dependence_fit = GIXOS_qxy_dependence(
    GIXOS_ana,
    GIXOS_ana["metadata"]["dependency"]["qz_selected"],
    row_window=3,
    fit_kappa=True,
)

### 7. Convert GIXOS to pseudo-reflectivity and structure factor

Apply the eCWM roughness factors at the selected `qxy0` positions to obtain
`R(Qz)`, `|Phi(Qz)|^2`, and the specular roughness factor `Psi_R(Qz)`.

In [ ]:
_ = GIXOS2R(
    GIXOS_ana,
    transmission_corr=True,
    footprint_effect=False,
    use_approx=True,
)

### 8. Export updated configuration YAML

Save the metadata back to YAML (in the same structure as the input config),
now including the fitted bending modulus `kappa`.

In [ ]:
configfilename = make_filename(GIXOS_ana["metadata"], suffix="cfg.yaml")
save_metadata_yaml(GIXOS_ana["metadata"], configfilename)
print("config written:", configfilename)

### 9. Export ORSO files

- `refl`  : pseudo-reflectivity R(Qz)
- `SF`    : structure factor |Phi(Qz)|^2
- `GIXOS` : background-corrected GIXOS (I0 * R*)

In [ ]:
_ = export_orso(GIXOS_ana, which="refl")
_ = export_orso(GIXOS_ana, which="SF")
_ = export_orso(GIXOS_ana, which="GIXOS")